# OpenCV Crash Course: 2026 Edition
## Module 9: OCR (Read Text & Numbers)
**Estimated time: 15 minutes**

- Read text with EasyOCR
- Read only the region you need (ROI) and get a number
- Clean bad images with OpenCV before OCR

> Meter readings, displays, number plates, labels: OCR + OpenCV preprocessing.

In [ ]:
# !pip install easyocr

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr

def show_grid(images, titles, figsize=(16, 4)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    for ax, img, t in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap='gray')
        ax.set_title(t, fontsize=11)
        ax.axis('off')
    plt.show()

# Test images made with OpenCV (no camera needed)
panel = np.full((240, 640, 3), 30, np.uint8)
cv2.putText(panel, "FLOW RATE", (40, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (200, 200, 200), 2)
cv2.putText(panel, "482.7", (60, 170), cv2.FONT_HERSHEY_DUPLEX, 3.0, (60, 255, 60), 5)
cv2.putText(panel, "m3/h", (450, 170), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (60, 255, 60), 3)

label = np.full((220, 520, 3), 235, np.uint8)
cv2.putText(label, "BATCH: A7-2026", (25, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (20, 20, 20), 3)
cv2.putText(label, "EXP 12/2027", (25, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (20, 20, 20), 3)
np.random.seed(0)
bad = (label * 0.35 + 40).astype(np.uint8)                                          # dark, low contrast
bad = np.clip(bad + np.random.normal(0, 18, bad.shape), 0, 255).astype(np.uint8)    # noisy

show_grid([panel, bad], ['Display panel', 'Bad-quality label'], figsize=(14, 3))

## 1. OCR in Two Lines

Returns `(box, text, confidence)` per line. Hindi + English: `easyocr.Reader(['hi', 'en'])`.

In [ ]:
reader = easyocr.Reader(['en'], gpu=False)
for box, text, conf in reader.readtext(panel):
    print(f"{text:<12} conf={conf:.2f}")

## 2. ROI to Number

Fixed camera = display always in the same place. **Crop first, then OCR.**
`allowlist` stops mistakes like `O` vs `0`.

In [ ]:
roi = panel[90:190, 40:440]
txt = reader.readtext(roi, allowlist='0123456789.', detail=0)
value = float(txt[0])
print("Flow:", value)
if value > 450:
    print("ALERT: flow above 450 m3/h")

## 3. Preprocess Bad Images

gray → denoise → CLAHE → upscale x2 → Otsu threshold (all from Module 2).

In [ ]:
gray  = cv2.cvtColor(bad, cv2.COLOR_BGR2GRAY)
clean = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(cv2.fastNlMeansDenoising(gray, h=15))
big   = cv2.resize(clean, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
_, binary = cv2.threshold(big, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
show_grid([gray, binary], ['Before', 'After preprocessing'], figsize=(12, 3))

for name, im in [('before', bad), ('after', binary)]:
    print(name, [(t, round(float(c), 2)) for _, t, c in reader.readtext(im)])

---
## Summary

| Task | Code |
|------|------|
| Reader | `easyocr.Reader(['en'], gpu=False)` |
| Read | `reader.readtext(img)` |
| Text only | `reader.readtext(img, detail=0)` |
| Digits only | `reader.readtext(roi, allowlist='0123456789.')` |

Other options: **PaddleOCR** (often most accurate), **Tesseract** (clean scanned text).

**Next:** Module 10, ONNX deployment